In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 19 — Local Product Docs Copilot
## Chat Over Internal or API Documentation

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 2 — Sample API Documentation

In [ ]:
from langchain.schema import Document

api_docs = [
    Document(page_content="""## Authentication
All API requests require a Bearer token in the Authorization header.

POST /api/v1/auth/token
Body: {"email": "user@example.com", "password": "..."}
Response: {"token": "eyJ...", "expires_in": 3600}

The token expires in 1 hour. Use the refresh endpoint to get a new token.
POST /api/v1/auth/refresh
Header: Authorization: Bearer <expired_token>
Response: {"token": "new_eyJ...", "expires_in": 3600}""",
        metadata={"section": "authentication", "version": "v1"}),

    Document(page_content="""## Users API
GET /api/v1/users — List all users (paginated, 50 per page)
GET /api/v1/users/:id — Get user by ID
POST /api/v1/users — Create user (requires admin role)
PATCH /api/v1/users/:id — Update user
DELETE /api/v1/users/:id — Soft delete user

Query Parameters:
- page (int): Page number, default 1
- per_page (int): Items per page, max 100
- search (string): Filter by name or email
- role (string): Filter by role (admin, user, viewer)""",
        metadata={"section": "users", "version": "v1"}),

    Document(page_content="""## Webhooks
POST /api/v1/webhooks — Register a webhook
Body: {"url": "https://...", "events": ["user.created", "user.updated"]}

Available events: user.created, user.updated, user.deleted,
document.created, document.shared.

Webhook payload format:
{"event": "user.created", "data": {...}, "timestamp": "2025-01-15T10:30:00Z"}

Retry policy: 3 attempts with exponential backoff (1s, 5s, 25s).
Webhooks are disabled after 10 consecutive failures.""",
        metadata={"section": "webhooks", "version": "v1"}),

    Document(page_content="""## Rate Limits
Default limits: 100 requests/minute per API key.
Bulk endpoints: 10 requests/minute.
Auth endpoints: 20 requests/minute.

Rate limit headers included in every response:
X-RateLimit-Limit: 100
X-RateLimit-Remaining: 95
X-RateLimit-Reset: 1705312800

When rate limited, respond with HTTP 429 and Retry-After header.""",
        metadata={"section": "rate_limits", "version": "v1"}),
]
print(f"Loaded {len(api_docs)} API documentation sections")

## Step 3 — Build Docs Index

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(api_docs)

vectorstore = Chroma.from_documents(chunks, embeddings,
    persist_directory="sample_data/docs_chroma", collection_name="api_docs")
print(f"Docs index: {len(chunks)} chunks")

## Step 4 — Developer Q&A

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

docs_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a developer support assistant for our API.
Answer technical questions using the documentation. Include:
- Exact endpoint URLs and HTTP methods
- Example request/response bodies
- Important headers or parameters
- Common gotchas or tips

Documentation:
{context}

Developer Question: {question}

Technical Answer:"""
)

docs_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": docs_prompt},
)

dev_questions = [
    "How do I authenticate my API requests?",
    "What's the rate limit and what happens when I exceed it?",
    "How do I set up a webhook for new user events?",
    "How do I search for users by email?",
]

for q in dev_questions:
    print(f"\nQ: {q}")
    result = docs_qa.invoke({"query": q})
    print(f"A: {result['result']}")

## Step 5 — Code Example Generator

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

codegen_prompt = ChatPromptTemplate.from_messages([
    ("system", "Generate a working Python code example using the requests library. "
     "Include error handling and comments."),
    ("human", "Generate Python code for: {task}\n\nAPI docs: {docs}")
])
codegen = codegen_prompt | llm | StrOutputParser()

# Get relevant docs and generate code
task = "Authenticate and list all admin users"
relevant = vectorstore.similarity_search(task, k=2)
docs_text = "\n".join([d.page_content for d in relevant])
code_example = codegen.invoke({"task": task, "docs": docs_text})
print("Generated Code Example:")
print(code_example)

## What You Learned
- **API documentation indexing** with section metadata
- **Developer Q&A** with endpoint-specific answers
- **Code example generation** from API docs